# 🌟 Gold Layer Pipeline — PM2.5 Forecasting (2023-2025)

**Purpose**: Transform Silver AQ data (3 complete years) into ML-ready Gold layer

**Input**: 
- Silver Air Quality: 2023, 2024, 2025 (complete years only)
- 79 Bangkok stations

**Output**:
- Gold layer with engineered features
- Train/Val/Test splits (chronological)
- Normalized features
- Ready for ML model training

**Pipeline Steps**:
1. Load Silver data (2023-2025)
2. Data quality assessment
3. Feature engineering (lags, rolling stats, temporal)
4. Create target variable (24h ahead forecast)
5. Chronological train/val/test split (70/15/15)
6. Normalization (using train stats only)
7. Save Gold layer

In [ ]:
from __future__ import annotations

import json
import sys
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import polars as pl

sys.path.insert(0, str(Path.cwd().parent.parent))

print("✅ Imports loaded")

## Configuration

In [ ]:
PROJECT_ROOT = Path.cwd().parent.parent

SILVER_AQ = PROJECT_ROOT / "data" / "silver" / "openmeteo_airquality"
GOLD_OUTPUT = PROJECT_ROOT / "data" / "gold"
STATIONS_PATH = PROJECT_ROOT / "data" / "stations" / "bangkok_stations.parquet"

TARGET_YEARS = [2023, 2024, 2025]
TARGET_COLUMN = "pm2_5_ugm3"
FORECAST_HORIZON = 24  # Predict 24 hours ahead

LAG_HOURS = [1, 2, 3, 6, 12, 24]
ROLLING_WINDOWS = [3, 6, 12, 24]

TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15

GOLD_OUTPUT.mkdir(parents=True, exist_ok=True)

print(f"✅ Configuration:")
print(f"   Years: {TARGET_YEARS}")
print(f"   Forecast horizon: {FORECAST_HORIZON}h")
print(f"   Lags: {LAG_HOURS}")
print(f"   Rolling windows: {ROLLING_WINDOWS}")
print(f"   Split: {TRAIN_RATIO}/{VAL_RATIO}/{TEST_RATIO}")

## Step 1: Load Silver Data (2023-2025)

In [ ]:
print("\n📂 Loading Silver AQ data...\n")

all_dfs = []

for year in TARGET_YEARS:
    year_path = SILVER_AQ / f"year={year}"
    
    if not year_path.exists():
        print(f"⚠️  Year {year}: Not found")
        continue
    
    year_dfs = []
    
    for month in range(1, 13):
        month_path = year_path / f"month={month:02d}"
        
        if not month_path.exists():
            continue
        
        files = [f for f in month_path.iterdir() if f.suffix == ".parquet" and not f.name.endswith(".md5")]
        
        for pf in files:
            try:
                df = pl.read_parquet(pf)
                
                # Standardize column names
                rename_map = {
                    "no2_ugm3": "nitrogen_dioxide_ugm3",
                    "o3_ugm3": "ozone_ugm3",
                    "so2_ugm3": "sulphur_dioxide_ugm3",
                    "co_ugm3": "carbon_monoxide_ugm3",
                }
                
                for old, new in rename_map.items():
                    if old in df.columns:
                        df = df.rename({old: new})
                
                # Standardize datetime
                if df.schema["timestamp_utc"] == pl.Datetime("ns", "UTC"):
                    df = df.with_columns(
                        pl.col("timestamp_utc").cast(pl.Datetime("us", "UTC"))
                    )
                
                # Ensure timezone
                if "timestamp_utc" in df.columns:
                    ts_dtype = df.schema["timestamp_utc"]
                    if ts_dtype.time_zone is None:
                        df = df.with_columns(
                            pl.col("timestamp_utc").dt.replace_time_zone("UTC")
                        )
                
                # Cast to Float64
                numeric_cols = [
                    "pm2_5_ugm3", "pm10_ugm3", "nitrogen_dioxide_ugm3",
                    "ozone_ugm3", "sulphur_dioxide_ugm3", "carbon_monoxide_ugm3"
                ]
                
                for col in numeric_cols:
                    if col in df.columns:
                        df = df.with_columns(pl.col(col).cast(pl.Float64))
                
                year_dfs.append(df)
                
            except Exception as e:
                print(f"⚠️  Error reading {pf.name}: {e}")
    
    if year_dfs:
        year_combined = pl.concat(year_dfs)
        all_dfs.append(year_combined)
        print(f"✅ Year {year}: {len(year_combined):,} rows | {year_combined['stationID'].n_unique()} stations")

df = pl.concat(all_dfs)
df = df.sort(["stationID", "timestamp_utc"])

print(f"\n✅ Total loaded: {len(df):,} rows")
print(f"   Date range: {df['timestamp_utc'].min()} → {df['timestamp_utc'].max()}")
print(f"   Stations: {df['stationID'].n_unique()}")
print(f"   Columns: {len(df.columns)}")

## Step 2: Data Quality Assessment

In [ ]:
print("\n🔍 Data Quality Report:\n")

# Check null values
null_counts = df.null_count()

print("Null counts:")
for col in ["pm2_5_ugm3", "pm10_ugm3", "nitrogen_dioxide_ugm3", "ozone_ugm3"]:
    if col in df.columns:
        nulls = null_counts[col][0]
        pct = (nulls / len(df)) * 100
        status = "✅" if pct < 5 else "⚠️" if pct < 20 else "❌"
        print(f"  {status} {col}: {nulls:,} ({pct:.2f}%)")

# Check PM2.5 statistics
pm25_stats = df["pm2_5_ugm3"].describe()
print(f"\nPM2.5 statistics:")
print(pm25_stats)

# Remove rows with null PM2.5 (our target)
initial_rows = len(df)
df = df.filter(pl.col("pm2_5_ugm3").is_not_null())
removed = initial_rows - len(df)

print(f"\n🔧 Removed {removed:,} rows with null PM2.5")
print(f"   Remaining: {len(df):,} rows")

## Step 3: Feature Engineering

### 3.1 Temporal Features

In [ ]:
print("\n📊 Adding temporal features...\n")

df = df.with_columns([
    pl.col("timestamp_utc").dt.hour().alias("hour"),
    pl.col("timestamp_utc").dt.day().alias("day"),
    pl.col("timestamp_utc").dt.month().alias("month"),
    pl.col("timestamp_utc").dt.weekday().alias("day_of_week"),
    pl.col("timestamp_utc").dt.year().alias("year"),
    (pl.col("timestamp_utc").dt.weekday() >= 5).alias("is_weekend"),
])

# Cyclical encoding
df = df.with_columns([
    (2 * np.pi * pl.col("hour") / 24).sin().alias("hour_sin"),
    (2 * np.pi * pl.col("hour") / 24).cos().alias("hour_cos"),
    (2 * np.pi * (pl.col("month") - 1) / 12).sin().alias("month_sin"),
    (2 * np.pi * (pl.col("month") - 1) / 12).cos().alias("month_cos"),
    (2 * np.pi * pl.col("day_of_week") / 7).sin().alias("dow_sin"),
    (2 * np.pi * pl.col("day_of_week") / 7).cos().alias("dow_cos"),
])

print(f"✅ Added temporal features")
print(f"   Current shape: {df.shape}")

### 3.2 Lag Features

In [ ]:
print(f"\n📊 Adding lag features: {LAG_HOURS}\n")

lag_exprs = []
for lag in LAG_HOURS:
    lag_exprs.append(
        pl.col(TARGET_COLUMN)
        .shift(lag)
        .over("stationID")
        .alias(f"{TARGET_COLUMN}_lag_{lag}h")
    )

df = df.with_columns(lag_exprs)

print(f"✅ Added {len(LAG_HOURS)} lag features")
print(f"   Current shape: {df.shape}")

### 3.3 Rolling Window Statistics

In [ ]:
print(f"\n📊 Adding rolling window features: {ROLLING_WINDOWS}\n")

rolling_exprs = []
for window in ROLLING_WINDOWS:
    rolling_exprs.extend([
        pl.col(TARGET_COLUMN).rolling_mean(window).over("stationID").alias(f"{TARGET_COLUMN}_rolling_mean_{window}h"),
        pl.col(TARGET_COLUMN).rolling_std(window).over("stationID").alias(f"{TARGET_COLUMN}_rolling_std_{window}h"),
        pl.col(TARGET_COLUMN).rolling_min(window).over("stationID").alias(f"{TARGET_COLUMN}_rolling_min_{window}h"),
        pl.col(TARGET_COLUMN).rolling_max(window).over("stationID").alias(f"{TARGET_COLUMN}_rolling_max_{window}h"),
    ])

df = df.with_columns(rolling_exprs)

print(f"✅ Added {len(ROLLING_WINDOWS) * 4} rolling features")
print(f"   Current shape: {df.shape}")

### 3.4 Rate of Change Features

In [ ]:
print("\n📊 Adding rate of change features\n")

df = df.with_columns([
    (pl.col(TARGET_COLUMN) - pl.col(TARGET_COLUMN).shift(1).over("stationID")).alias(f"{TARGET_COLUMN}_diff_1h"),
    (pl.col(TARGET_COLUMN) - pl.col(TARGET_COLUMN).shift(24).over("stationID")).alias(f"{TARGET_COLUMN}_diff_24h"),
    ((pl.col(TARGET_COLUMN) - pl.col(TARGET_COLUMN).shift(1).over("stationID")) / 
     pl.col(TARGET_COLUMN).shift(1).over("stationID") * 100).alias(f"{TARGET_COLUMN}_pct_change_1h"),
])

print(f"✅ Added rate of change features")
print(f"   Current shape: {df.shape}")

### 3.5 Create Target Variable (Future PM2.5)

In [ ]:
print(f"\n🎯 Creating target: PM2.5 at t+{FORECAST_HORIZON}h\n")

df = df.with_columns(
    pl.col(TARGET_COLUMN)
    .shift(-FORECAST_HORIZON)
    .over("stationID")
    .alias(f"target_{TARGET_COLUMN}_{FORECAST_HORIZON}h")
)

print(f"✅ Target variable created")
print(f"   Current shape: {df.shape}")

# Remove rows with null features or target
before_dropna = len(df)
df = df.drop_nulls()
after_dropna = len(df)

print(f"\n🔧 Dropped {before_dropna - after_dropna:,} rows with null values")
print(f"   Final dataset: {after_dropna:,} rows")

## Step 4: Train/Val/Test Split (Chronological)

In [ ]:
print("\n📊 Creating chronological splits...\n")

df = df.sort("timestamp_utc")

n = len(df)
train_end = int(n * TRAIN_RATIO)
val_end = int(n * (TRAIN_RATIO + VAL_RATIO))

train_df = df[:train_end]
val_df = df[train_end:val_end]
test_df = df[val_end:]

print(f"Train: {len(train_df):,} rows | {train_df['timestamp_utc'].min()} → {train_df['timestamp_utc'].max()}")
print(f"Val:   {len(val_df):,} rows | {val_df['timestamp_utc'].min()} → {val_df['timestamp_utc'].max()}")
print(f"Test:  {len(test_df):,} rows | {test_df['timestamp_utc'].min()} → {test_df['timestamp_utc'].max()}")

print(f"\n✅ Splits created (no data leakage - chronological order preserved)")

## Step 5: Normalization (Using Train Stats Only)

In [ ]:
print("\n🔧 Normalizing features (z-score)...\n")

# Identify feature columns
exclude_cols = [
    "stationID", "timestamp_utc", "timestamp_unix_ms",
    "data_source", "ingestion_timestamp_utc", "load_id",
    "pipeline_version", "record_hash", "lat", "lon",
    f"target_{TARGET_COLUMN}_{FORECAST_HORIZON}h"
]

feature_cols = [c for c in df.columns if c not in exclude_cols]

print(f"Features to normalize: {len(feature_cols)}")

# Compute stats from TRAIN set only
norm_stats = {}

for col in feature_cols:
    if col not in train_df.columns:
        continue
    
    mean = train_df[col].mean()
    std = train_df[col].std()
    
    if std == 0 or std is None or np.isnan(std):
        continue
    
    norm_stats[col] = {"mean": float(mean), "std": float(std)}
    
    # Apply to all splits
    train_df = train_df.with_columns(((pl.col(col) - mean) / std).alias(col))
    val_df = val_df.with_columns(((pl.col(col) - mean) / std).alias(col))
    test_df = test_df.with_columns(((pl.col(col) - mean) / std).alias(col))

print(f"\n✅ Normalized {len(norm_stats)} features")
print(f"   Sample stats: {list(norm_stats.keys())[:5]}")

## Step 6: Save Gold Layer

In [ ]:
print("\n💾 Saving Gold layer...\n")

train_path = GOLD_OUTPUT / "train.parquet"
val_path = GOLD_OUTPUT / "val.parquet"
test_path = GOLD_OUTPUT / "test.parquet"

train_df.write_parquet(train_path, compression="snappy")
val_df.write_parquet(val_path, compression="snappy")
test_df.write_parquet(test_path, compression="snappy")

print(f"✅ Train: {train_path.name} ({len(train_df):,} rows)")
print(f"✅ Val:   {val_path.name} ({len(val_df):,} rows)")
print(f"✅ Test:  {test_path.name} ({len(test_df):,} rows)")

# Save normalization stats
stats_path = GOLD_OUTPUT / "normalization_stats.json"
with open(stats_path, "w") as f:
    json.dump(norm_stats, f, indent=2)

print(f"✅ Stats: {stats_path.name}")

# Save metadata
metadata = {
    "pipeline_version": "1.0.0",
    "created_at": datetime.utcnow().isoformat() + "Z",
    "source_years": TARGET_YEARS,
    "total_rows": len(df),
    "train_rows": len(train_df),
    "val_rows": len(val_df),
    "test_rows": len(test_df),
    "num_features": len(feature_cols),
    "forecast_horizon": FORECAST_HORIZON,
    "target_column": f"target_{TARGET_COLUMN}_{FORECAST_HORIZON}h",
    "lag_features": LAG_HOURS,
    "rolling_windows": ROLLING_WINDOWS,
}

metadata_path = GOLD_OUTPUT / "pipeline_metadata.json"
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"✅ Metadata: {metadata_path.name}")

## Step 7: Summary & Validation

In [ ]:
print("\n" + "=" * 100)
print(" " * 35 + "✅ GOLD LAYER COMPLETE")
print("=" * 100)

print(f"\n📊 Dataset Summary:")
print(f"   Source years: {TARGET_YEARS}")
print(f"   Total samples: {len(df):,}")
print(f"   Features: {len(feature_cols)}")
print(f"   Target: {TARGET_COLUMN} at t+{FORECAST_HORIZON}h")

print(f"\n📊 Split Summary:")
print(f"   Train: {len(train_df):,} ({len(train_df)/len(df)*100:.1f}%)")
print(f"   Val:   {len(val_df):,} ({len(val_df)/len(df)*100:.1f}%)")
print(f"   Test:  {len(test_df):,} ({len(test_df)/len(df)*100:.1f}%)")

print(f"\n📁 Output files:")
print(f"   {GOLD_OUTPUT}/train.parquet")
print(f"   {GOLD_OUTPUT}/val.parquet")
print(f"   {GOLD_OUTPUT}/test.parquet")
print(f"   {GOLD_OUTPUT}/normalization_stats.json")
print(f"   {GOLD_OUTPUT}/pipeline_metadata.json")

print("\n" + "=" * 100)
print("🎉 Ready for ML model training!")
print("=" * 100)

## Step 8: Quick Data Inspection

In [ ]:
# Show sample of training data
print("\n📋 Training Data Sample:\n")
print(train_df.head(10))

# Show feature columns
print(f"\n📊 Feature Columns ({len(feature_cols)}):")
for i, col in enumerate(feature_cols[:20], 1):
    print(f"   {i:2d}. {col}")
if len(feature_cols) > 20:
    print(f"   ... and {len(feature_cols) - 20} more")

---

## Next: Model Training

Gold layer is ready! Next steps:

1. **Load Gold data** in model training notebook
2. **Build ST-UNN model** (Spatio-Temporal U-Net)
3. **Train with 3 years** of data (2023-2025)
4. **Evaluate** on chronological test set
5. **Deploy** for real-time forecasting

See: `notebooks/modeling/model_training.ipynb`